# Load Project Configuration

This cell loads the centralized project configuration notebook.

Loaded configurations:
- Catalog and schemas
- API endpoints
- Audit tables
- Naming conventions
- Pipeline metadata
- Environment variables

In [0]:
%run ../99_utils/utils_config

In [0]:
%run ../99_utils/utils_api_client

In [0]:
%run ../99_utils/utils_pagination

In [0]:
%run ../99_utils/utils_hash

In [0]:
%run ../99_utils/utils_legislature

In [0]:
%run ../99_utils/utils_logger

In [0]:
%run ../99_utils/utils_table_logger

PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: 5e4d23ac-2283-436a-9826-56bfc24ac71c


PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: f8cc7b3d-8dca-4d61-9081-d647be4e4369


PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: d7d820c3-f603-4cca-b6d3-40e3195d8839


PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: ac2ab318-9819-459c-ae4e-2ba645285c2f


PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: 3d090341-18de-40a5-876f-97923058e6d5


PROJECT CONFIGURATION LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: 119851f2-1ecf-4c47-8b9b-57350c4805f2


utils_config loaded successfully.


utils_config loaded successfully.


utils_config loaded successfully.


utils_logger loaded successfully.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 Bronze — Deputados
# MAGIC
# MAGIC **Notebook:** `01_bronze_deputados`
# MAGIC
# MAGIC Extracts deputy data from the Câmara dos Deputados Open Data API for the supported legislatures and persists the raw records into the Bronze layer.
# MAGIC
# MAGIC ## Source Endpoint
# MAGIC `/deputados`
# MAGIC
# MAGIC ## Target Table
# MAGIC `brazil_legislative_analytics.bronze.br_deputados`
# MAGIC
# MAGIC ## Documentation Standard
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and documentation are written in English.

# COMMAND ----------

# MAGIC %run ../99_utils/utils_config

# COMMAND ----------

# MAGIC %run ../99_utils/utils_api_client

# COMMAND ----------

# MAGIC %run ../99_utils/utils_pagination

# COMMAND ----------

# MAGIC %run ../99_utils/utils_hash

# COMMAND ----------

# MAGIC %run ../99_utils/utils_legislature

# COMMAND ----------

# MAGIC %run ../99_utils/utils_logger

# COMMAND ----------

# MAGIC %run ../99_utils/utils_table_logger

# COMMAND ----------

from datetime import datetime
import json
import uuid

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
)

# COMMAND ----------

print("=" * 90)
print("BRAZIL LEGISLATIVE ANALYTICS MEDALLION")
print("01 - BRONZE DEPUTADOS")
print("=" * 90)
print(f"Execution Timestamp: {datetime.now()}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# NOTEBOOK CONFIGURATION
# ============================================================

NOTEBOOK_NAME = "01_bronze_deputados"
LAYER_NAME = "bronze"
ENTITY_NAME = "deputados"

SOURCE_ENDPOINT = API_ENDPOINTS["deputados"]

TARGET_TABLE = get_bronze_table(
    BRONZE_TABLES["deputados"]
)

LOAD_TYPE = LOAD_TYPE_FULL

execution_id = str(uuid.uuid4())
started_at = datetime.now()

logger = get_logger(
    logger_name=NOTEBOOK_NAME,
    layer_name=LAYER_NAME,
)

# COMMAND ----------

# ============================================================
# INGESTION CONFIGURATION
# ============================================================

PAGE_SIZE = 100
MAX_PAGES = None

LEGISLATURE_IDS = get_supported_legislatures()

BASE_PARAMS = {
    "ordem": "ASC",
    "ordenarPor": "id",
}

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Start Pipeline Log

# COMMAND ----------

write_pipeline_log(
    log_id=str(uuid.uuid4()),
    execution_id=execution_id,
    notebook_name=NOTEBOOK_NAME,
    layer_name=LAYER_NAME,
    entity_name=ENTITY_NAME,
    target_table=TARGET_TABLE,
    status=EXECUTION_STATUS_STARTED,
    message="Bronze deputados ingestion started.",
    started_at=started_at,
    finished_at=None,
    duration_seconds=None,
    records_read=None,
    records_written=None,
)

log_info(
    pipeline_logger=logger,
    message="Starting deputados ingestion.",
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Extract API Records by Legislature

# COMMAND ----------

try:

    deputy_records = []

    for legislature_id in LEGISLATURE_IDS:

        log_info(
            pipeline_logger=logger,
            message=(
                f"Starting deputados extraction "
                f"| legislature_id={legislature_id}"
            ),
        )

        legislature_params = dict(BASE_PARAMS)
        legislature_params["idLegislatura"] = legislature_id

        legislature_records = collect_pages(
            endpoint_path=SOURCE_ENDPOINT,
            base_params=legislature_params,
            page_size=PAGE_SIZE,
            max_pages=MAX_PAGES,
        )

        for record in legislature_records:
            record["idLegislatura"] = legislature_id

        deputy_records.extend(legislature_records)

    records_read = len(deputy_records)

    log_info(
        pipeline_logger=logger,
        message=(
            f"Deputy records extracted: "
            f"{records_read}"
        ),
    )

except Exception as error:

    finished_at = datetime.now()
    duration_seconds = (
        finished_at - started_at
    ).total_seconds()

    write_pipeline_log(
        log_id=str(uuid.uuid4()),
        execution_id=execution_id,
        notebook_name=NOTEBOOK_NAME,
        layer_name=LAYER_NAME,
        entity_name=ENTITY_NAME,
        target_table=TARGET_TABLE,
        status=EXECUTION_STATUS_FAILED,
        message=(
            f"Failed during API extraction "
            f"| error={str(error)}"
        ),
        started_at=started_at,
        finished_at=finished_at,
        duration_seconds=duration_seconds,
        records_read=None,
        records_written=None,
    )

    log_error(
        pipeline_logger=logger,
        message="Deputados extraction failed.",
        error=error,
    )

    raise error

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Prepare Bronze Records

# COMMAND ----------

bronze_rows = []
ingestion_timestamp = datetime.now()

for deputy_record in deputy_records:

    raw_json_payload = json.dumps(
        deputy_record,
        ensure_ascii=False,
    )

    bronze_rows.append({
        "dep_id_legislatura": str(
            deputy_record.get("idLegislatura")
        ),
        "dep_id_deputado": str(
            deputy_record.get("id")
        ),
        "dep_tx_nome": deputy_record.get(
            "nome"
        ),
        "dep_tx_sigla_partido": deputy_record.get(
            "siglaPartido"
        ),
        "dep_tx_sigla_uf": deputy_record.get(
            "siglaUf"
        ),
        "dep_tx_url_foto": deputy_record.get(
            "urlFoto"
        ),
        "dep_tx_email": deputy_record.get(
            "email"
        ),
        "dep_tx_payload_json": raw_json_payload,
        "aud_id_execucao": execution_id,
        "aud_dh_ingestao": ingestion_timestamp,
        "aud_tx_endpoint_origem": SOURCE_ENDPOINT,
        "aud_tx_sistema_origem": "camara_api",
        "aud_tx_versao_pipeline": PROJECT_VERSION,
        "aud_tx_tipo_carga": LOAD_TYPE,
    })

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Create Bronze DataFrame

# COMMAND ----------

bronze_schema = StructType([
    StructField("dep_id_legislatura", StringType(), True),
    StructField("dep_id_deputado", StringType(), True),
    StructField("dep_tx_nome", StringType(), True),
    StructField("dep_tx_sigla_partido", StringType(), True),
    StructField("dep_tx_sigla_uf", StringType(), True),
    StructField("dep_tx_url_foto", StringType(), True),
    StructField("dep_tx_email", StringType(), True),
    StructField("dep_tx_payload_json", StringType(), True),
    StructField("aud_id_execucao", StringType(), True),
    StructField("aud_dh_ingestao", TimestampType(), True),
    StructField("aud_tx_endpoint_origem", StringType(), True),
    StructField("aud_tx_sistema_origem", StringType(), True),
    StructField("aud_tx_versao_pipeline", StringType(), True),
    StructField("aud_tx_tipo_carga", StringType(), True),
])

bronze_df = spark.createDataFrame(
    bronze_rows,
    bronze_schema,
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Add Record Hash

# COMMAND ----------

bronze_df = add_hash(
    dataframe=bronze_df,
    columns=[
        "dep_id_legislatura",
        "dep_id_deputado",
        "dep_tx_payload_json",
    ],
    hash_column="aud_tx_hash_registro",
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Persist Bronze Table

# COMMAND ----------

bronze_df.write.format(
    "delta"
).mode(
    "overwrite"
).option(
    "overwriteSchema",
    "true"
).saveAsTable(
    TARGET_TABLE
)

records_written = bronze_df.count()

log_info(
    pipeline_logger=logger,
    message=(
        f"Bronze table persisted successfully "
        f"| records_written={records_written}"
    ),
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Apply Table Comment

# COMMAND ----------

spark.sql(f"""
COMMENT ON TABLE {TARGET_TABLE}
IS '
Raw deputy ingestion table from Câmara dos Deputados Open Data API.

This Bronze table preserves deputy records extracted by supported legislature.

Main characteristics:
- raw ingestion fidelity
- original payload preservation
- legislature traceability
- ingestion metadata
- record hash support
- auditability

Source endpoint:
- /deputados
'
""")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Display Bronze Data

# COMMAND ----------

display(
    bronze_df.limit(20)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Final Pipeline Log

# COMMAND ----------

finished_at = datetime.now()

duration_seconds = (
    finished_at - started_at
).total_seconds()

write_pipeline_log(
    log_id=str(uuid.uuid4()),
    execution_id=execution_id,
    notebook_name=NOTEBOOK_NAME,
    layer_name=LAYER_NAME,
    entity_name=ENTITY_NAME,
    target_table=TARGET_TABLE,
    status=EXECUTION_STATUS_SUCCESS,
    message=(
        f"Bronze deputados ingestion completed successfully "
        f"| records_read={records_read} "
        f"| records_written={records_written}"
    ),
    started_at=started_at,
    finished_at=finished_at,
    duration_seconds=duration_seconds,
    records_read=records_read,
    records_written=records_written,
)

log_success(
    pipeline_logger=logger,
    message=(
        f"Bronze deputados ingestion completed "
        f"| duration_seconds={duration_seconds}"
    ),
)

# COMMAND ----------

print("=" * 90)
print("BRONZE DEPUTADOS COMPLETED")
print("=" * 90)
print(f"Target Table: {TARGET_TABLE}")
print(f"Legislatures: {LEGISLATURE_IDS}")
print(f"Records Read: {records_read}")
print(f"Records Written: {records_written}")
print(f"Execution Duration: {duration_seconds}")
print("=" * 90)

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
01 - BRONZE DEPUTADOS
Execution Timestamp: 2026-05-20 05:07:42.193304


2026-05-20 05:07:43 | INFO | BRONZE | bronze.01_bronze_deputados | Starting deputados ingestion.
2026-05-20 05:07:43 | INFO | BRONZE | bronze.01_bronze_deputados | Starting deputados extraction | legislature_id=56


[INFO] Page collected | endpoint=/deputados | page=1 | page_records=100 | total_records=100
[INFO] Page collected | endpoint=/deputados | page=2 | page_records=100 | total_records=200
[INFO] Page collected | endpoint=/deputados | page=3 | page_records=100 | total_records=300
[INFO] Page collected | endpoint=/deputados | page=4 | page_records=100 | total_records=400
[INFO] Page collected | endpoint=/deputados | page=5 | page_records=100 | total_records=500
[INFO] Page collected | endpoint=/deputados | page=6 | page_records=100 | total_records=600
[INFO] Page collected | endpoint=/deputados | page=7 | page_records=100 | total_records=700
[INFO] Page collected | endpoint=/deputados | page=8 | page_records=100 | total_records=800
[INFO] Page collected | endpoint=/deputados | page=9 | page_records=100 | total_records=900
[INFO] Page collected | endpoint=/deputados | page=10 | page_records=100 | total_records=1000


2026-05-20 05:08:40 | INFO | BRONZE | bronze.01_bronze_deputados | Starting deputados extraction | legislature_id=57


[INFO] Page collected | endpoint=/deputados | page=11 | page_records=73 | total_records=1073
[INFO] Pagination finished with partial last page | endpoint=/deputados | page=11
[INFO] Page collected | endpoint=/deputados | page=1 | page_records=100 | total_records=100
[INFO] Page collected | endpoint=/deputados | page=2 | page_records=100 | total_records=200
[INFO] Page collected | endpoint=/deputados | page=3 | page_records=100 | total_records=300
[INFO] Page collected | endpoint=/deputados | page=4 | page_records=100 | total_records=400
[INFO] Page collected | endpoint=/deputados | page=5 | page_records=100 | total_records=500
[INFO] Page collected | endpoint=/deputados | page=6 | page_records=100 | total_records=600
[INFO] Page collected | endpoint=/deputados | page=7 | page_records=100 | total_records=700
[INFO] Page collected | endpoint=/deputados | page=8 | page_records=100 | total_records=800


...

utils_hash loaded successfully.


utils_api_client loaded successfully.


utils_legislature loaded successfully.


utils_table_logger loaded successfully.


utils_pagination loaded successfully.
